In [3]:
from pathlib import Path
import subprocess
import json
import shutil
from datetime import datetime

# ============================================================
# Update conda/pip export files for every environment folder
#
# Expected structure:
#
# project_root/
#   Meta Scripts/
#     update_environment_exports.ipynb
#   environments/
#     cellpose/
#     yolo-seg/
#     update_environment/
#
# Each folder name should match the conda environment name.
# ============================================================

# ----------------------------
# Settings
# ----------------------------

REQUIRED_SELF_ENV = "update_environment"

EXPORT_ENVIRONMENT_YML = True
EXPORT_CONDA_LIST = True
EXPORT_CONDA_EXPLICIT = True
EXPORT_PIP_FREEZE = True


# ----------------------------
# Locate project directories
# ----------------------------

cwd = Path.cwd()

if cwd.name == "Meta Scripts":
  project_root = cwd.parent
else:
  project_root = cwd

environments_dir = project_root / "environments"

if not environments_dir.exists():
  raise FileNotFoundError(f"Could not find environments folder at: {environments_dir}")

print(f"Project root: {project_root}")
print(f"Environments folder: {environments_dir}")


# ----------------------------
# Find conda executable
# ----------------------------

def find_conda_executable():
  conda_path = shutil.which("conda")

  if conda_path is not None:
    return conda_path

  home = Path.home()

  possible_paths = [
    home / "miniconda3" / "Scripts" / "conda.exe",
    home / "anaconda3" / "Scripts" / "conda.exe",
    home / "Miniconda3" / "Scripts" / "conda.exe",
    home / "Anaconda3" / "Scripts" / "conda.exe",
    Path("C:/ProgramData/miniconda3/Scripts/conda.exe"),
    Path("C:/ProgramData/anaconda3/Scripts/conda.exe"),
    Path("C:/ProgramData/Miniconda3/Scripts/conda.exe"),
    Path("C:/ProgramData/Anaconda3/Scripts/conda.exe"),
  ]

  for path in possible_paths:
    if path.exists():
      return str(path)

  raise FileNotFoundError(
    "Could not find conda.exe.\n\n"
    "Try opening VSCode from Anaconda Prompt or Miniconda Prompt:\n\n"
    'cd "C:\\path\\to\\your\\project"\n'
    "code ."
  )


CONDA_EXE = find_conda_executable()
print(f"Using conda executable: {CONDA_EXE}")


# ----------------------------
# Helper functions
# ----------------------------

def run_command(command):
  if command[0] == "conda":
    command = [CONDA_EXE] + command[1:]

  result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    shell=False
  )

  if result.returncode != 0:
    raise RuntimeError(
      "Command failed:\n"
      + " ".join(str(part) for part in command)
      + "\n\nSTDOUT:\n"
      + result.stdout
      + "\n\nSTDERR:\n"
      + result.stderr
    )

  return result.stdout


def print_progress(current, total, label="", bar_length=30):
  if total == 0:
    percent = 100
    filled_length = bar_length
  else:
    percent = int(100 * current / total)
    filled_length = int(bar_length * current / total)

  bar = "█" * filled_length + "-" * (bar_length - filled_length)

  print(f"\r[{bar}] {percent:3d}%  {current}/{total}  {label}", end="")

  if current == total:
    print()


def write_text_file(path, text):
  path.write_text(text, encoding="utf-8")


# ----------------------------
# Get available conda environments
# ----------------------------

print("\nReading available conda environments...")

conda_info_raw = run_command(["conda", "env", "list", "--json"])
conda_info = json.loads(conda_info_raw)

available_env_paths = [Path(path) for path in conda_info.get("envs", [])]
available_env_names = {path.name for path in available_env_paths}

print("\nAvailable conda environments:")
for name in sorted(available_env_names):
  print(f"  - {name}")


# ----------------------------
# Get environment folders
# ----------------------------

env_folders = sorted(
  [folder for folder in environments_dir.iterdir() if folder.is_dir()],
  key=lambda path: path.name.lower()
)

print("\nEnvironment folders found:")
for folder in env_folders:
  print(f"  - {folder.name}")


# ----------------------------
# Sanity checks
# ----------------------------

if REQUIRED_SELF_ENV not in available_env_names:
  print(
    f"\nWARNING: Conda environment '{REQUIRED_SELF_ENV}' was not found.\n"
    f"If this notebook is supposed to run from that environment, create it first."
  )

if not (environments_dir / REQUIRED_SELF_ENV).exists():
  print(
    f"\nWARNING: Folder '{environments_dir / REQUIRED_SELF_ENV}' does not exist.\n"
    f"Create it if you want to export the update_environment environment too."
  )


# ----------------------------
# Calculate total progress steps
# ----------------------------

exports_per_env = sum([
  EXPORT_ENVIRONMENT_YML,
  EXPORT_CONDA_LIST,
  EXPORT_CONDA_EXPLICIT,
  EXPORT_PIP_FREEZE
])

valid_env_folders = [
  folder for folder in env_folders
  if folder.name in available_env_names
]

total_steps = len(valid_env_folders) * exports_per_env
current_step = 0

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
summary = []


# ----------------------------
# Export each environment
# ----------------------------

print("\nStarting exports...\n")

for env_folder in env_folders:
  env_name = env_folder.name

  print("\n" + "=" * 80)
  print(f"Checking environment: {env_name}")

  if env_name not in available_env_names:
    print(f"Skipping: no conda environment named '{env_name}' was found.")
    summary.append((env_name, "skipped - conda env not found"))
    continue

  print(f"Exporting files into: {env_folder}")

  try:
    # --------------------------------------------------------
    # 1. environment.yml
    # --------------------------------------------------------

    if EXPORT_ENVIRONMENT_YML:
      current_step += 1
      print_progress(current_step, total_steps, f"{env_name}: environment.yml")

      environment_yml = run_command([
        "conda", "env", "export",
        "-n", env_name,
        "--no-builds"
      ])

      cleaned_lines = []

      for line in environment_yml.splitlines():
        if not line.startswith("prefix:"):
          cleaned_lines.append(line)

      environment_yml = "\n".join(cleaned_lines) + "\n"

      write_text_file(
        env_folder / "environment.yml",
        environment_yml
      )

    # --------------------------------------------------------
    # 2. conda_list.txt
    # --------------------------------------------------------

    if EXPORT_CONDA_LIST:
      current_step += 1
      print_progress(current_step, total_steps, f"{env_name}: conda_list.txt")

      conda_list = run_command([
        "conda", "list",
        "-n", env_name
      ])

      write_text_file(
        env_folder / "conda_list.txt",
        f"# Updated: {timestamp}\n"
        f"# Conda environment: {env_name}\n\n"
        f"{conda_list}"
      )

    # --------------------------------------------------------
    # 3. conda_explicit.txt
    # --------------------------------------------------------

    if EXPORT_CONDA_EXPLICIT:
      current_step += 1
      print_progress(current_step, total_steps, f"{env_name}: conda_explicit.txt")

      conda_explicit = run_command([
        "conda", "list",
        "-n", env_name,
        "--explicit"
      ])

      write_text_file(
        env_folder / "conda_explicit.txt",
        f"# Updated: {timestamp}\n"
        f"# Conda environment: {env_name}\n\n"
        f"{conda_explicit}"
      )

    # --------------------------------------------------------
    # 4. pip_freeze.txt
    # --------------------------------------------------------

    if EXPORT_PIP_FREEZE:
      current_step += 1
      print_progress(current_step, total_steps, f"{env_name}: pip_freeze.txt")

      pip_freeze = run_command([
        "conda", "run",
        "-n", env_name,
        "python", "-m", "pip", "freeze"
      ])

      write_text_file(
        env_folder / "pip_freeze.txt",
        f"# Updated: {timestamp}\n"
        f"# Conda environment: {env_name}\n\n"
        f"{pip_freeze}"
      )

    print(f"\nFinished exporting: {env_name}")

    summary.append((env_name, "updated"))

  except Exception as error:
    print(f"\nERROR while exporting '{env_name}':")
    print(error)
    summary.append((env_name, "error"))


# ----------------------------
# Final summary
# ----------------------------

print("\n" + "=" * 80)
print("Export summary:")
print(f"Finished at: {timestamp}")

for env_name, status in summary:
  print(f"  - {env_name}: {status}")

print("\nDone.")

Project root: d:\2026\aspire
Environments folder: d:\2026\aspire\environments
Using conda executable: C:\ProgramData\anaconda3\condabin\conda.BAT

Available conda environments:
  - anaconda3
  - annotations
  - cellpose
  - conversion
  - update_environment
  - yolo-seg

Environment folders found:
  - annotations
  - cellpose
  - conversion
  - update_environment
  - yolo-seg

Checking environment: annotations
Exporting environment files into: d:\2026\aspire\environments\annotations
Updated:
  - environment.yml
  - conda_list.txt
  - conda_explicit.txt
  - pip_freeze.txt

Checking environment: cellpose
Exporting environment files into: d:\2026\aspire\environments\cellpose
Updated:
  - environment.yml
  - conda_list.txt
  - conda_explicit.txt
  - pip_freeze.txt

Checking environment: conversion
Exporting environment files into: d:\2026\aspire\environments\conversion
Updated:
  - environment.yml
  - conda_list.txt
  - conda_explicit.txt
  - pip_freeze.txt

Checking environment: update_en